In [1]:
import re
import json
import numpy as np
import trimesh
from pathlib import Path
from scipy.spatial.transform import Rotation

In [2]:
PROJECT_ROOT = Path(".").resolve().parent

GFILES_DIR       = PROJECT_ROOT / "environments" / "gfiles"
MESH_DATASET     = PROJECT_ROOT / "environments" / "mesh_dataset"
CAMERA_INFO_PATH = PROJECT_ROOT / "outputs" / "camera_info.json"
OUT_DIR          = PROJECT_ROOT / "yolo_dataset"

CAM_WIDTH  = 1920
CAM_HEIGHT = 1920

CLASS_NAMES = [
    "airplane", "bed", "car", "chair", "cone",
    "firefighterFigure", "guitar", "lamp", "laptop",
    "monitor", "piano", "stairs", "table", "vase", "xbox",
]
CLASS_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

GOAL_PAT = re.compile(
    r'^(goal_(?!pose_)\w+)\s*(?:\(\w+\))?\s*:\s*\{([^}]*)\}',
    re.MULTILINE,
)

In [3]:
def parse_goal_object(gfile_path: Path) -> dict | None:
    text = gfile_path.read_text()
    m = GOAL_PAT.search(text)
    if not m:
        return None
    body = m.group(2)

    pm = re.search(r'pose:\s*\[([^\]]+)\]', body)
    sm = re.search(r'meshscale:\s*\[([^\]]+)\]', body)
    mm = re.search(r'mesh:\s*<([^>]+)>', body)
    if not (pm and sm and mm):
        return None

    pose      = [float(x) for x in pm.group(1).split(',')]
    meshscale = float(sm.group(1).split(',')[0])
    mesh_rel  = mm.group(1).strip()

    idx = mesh_rel.find('mesh_dataset/')
    if idx == -1:
        return None
    mesh_sub = mesh_rel[idx + len('mesh_dataset/'):]
    category = mesh_sub.split('/')[0]

    if category not in CLASS_ID:
        return None

    return dict(pose=pose, meshscale=meshscale, mesh_sub=mesh_sub, category=category)


def load_mesh_verts(mesh_sub: str, max_verts: int = 3000) -> np.ndarray | None:
    path = MESH_DATASET / mesh_sub
    if not path.exists():
        return None
    try:
        mesh  = trimesh.load(str(path), force='mesh', process=False)
        verts = np.array(mesh.vertices, dtype=np.float64)
        if len(verts) > max_verts:
            rng   = np.random.default_rng(seed=0)
            verts = verts[rng.choice(len(verts), max_verts, replace=False)]
        return verts
    except Exception as e:
        print(f"  [warn] trimesh: {e}")
        return None


def pose7d_to_matrix(pose: list) -> np.ndarray:
    """RAI [x, y, z, qw, qx, qy, qz] → 4x4 matrix."""
    qw, qx, qy, qz = pose[3], pose[4], pose[5], pose[6]
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = pose[:3]
    return T


def compute_yolo_bbox(verts, meshscale, T_world_obj, K, T_world_cam,
                      W, H, min_area=400) -> tuple | None:
    scaled      = verts * meshscale
    hom         = np.hstack([scaled, np.ones((len(scaled), 1))])
    verts_world = (T_world_obj @ hom.T).T[:, :3]

    T_cam_world = np.linalg.inv(T_world_cam)
    hom_w       = np.hstack([verts_world, np.ones((len(verts_world), 1))])
    verts_cam   = (T_cam_world @ hom_w.T).T[:, :3]

    front = verts_cam[:, 2] > 0.01
    if not front.any():
        return None
    vc    = verts_cam[front]
    pts   = (K @ vc.T).T
    pts2d = pts[:, :2] / pts[:, 2:3]

    in_img = ((pts2d[:, 0] >= 0) & (pts2d[:, 0] < W) &
              (pts2d[:, 1] >= 0) & (pts2d[:, 1] < H))
    if not in_img.any():
        return None

    x1 = float(np.clip(pts2d[:, 0].min(), 0, W - 1))
    y1 = float(np.clip(pts2d[:, 1].min(), 0, H - 1))
    x2 = float(np.clip(pts2d[:, 0].max(), 0, W - 1))
    y2 = float(np.clip(pts2d[:, 1].max(), 0, H - 1))

    if (x2 - x1) * (y2 - y1) < min_area:
        return None

    return ((x1 + x2) / 2 / W,
            (y1 + y2) / 2 / H,
            (x2 - x1) / W,
            (y2 - y1) / H)

In [ ]:
with open(CAMERA_INFO_PATH) as f:
    camera_info = json.load(f)

cameras = {}
for cam_name, info in camera_info.items():
    cameras[cam_name] = {
        "K":           np.array(info["K"]),
        "T_world_cam": np.array(info["T_world_cam"]),
    }

print(f"Loaded {len(cameras)} cameras: {list(cameras.keys())}")

gfiles = sorted(
    GFILES_DIR.glob("env_*.g"),
    key=lambda p: int(re.search(r'\d+', p.stem).group()),
)
n_total = len(gfiles)
n_train = int(n_total * 0.8)
print(f"Found {n_total} scenes → {n_train} train / {n_total - n_train} val")

for split in ("train", "val"):
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

Loaded 5 cameras: ['cam_dim_0', 'cam_dim_1', 'cam_dim_2', 'cam_dim_3', 'cam_dim_4']
Found 80 scenes → 64 train / 16 val


In [5]:
stats = {"ok": 0, "no_goal": 0, "no_mesh": 0, "no_bbox": 0}

for env_idx, gfile in enumerate(gfiles):
    split    = "train" if env_idx < n_train else "val"
    env_name = gfile.stem   # e.g. "env_4"

    goal = parse_goal_object(gfile)
    if goal is None:
        print(f"  [{env_name}] no recognised goal object — skipping")
        stats["no_goal"] += 1
        continue

    verts = load_mesh_verts(goal["mesh_sub"])
    if verts is None:
        print(f"  [{env_name}] mesh not found ({goal['mesh_sub']}) — empty labels")
        stats["no_mesh"] += 1

    class_id    = CLASS_ID[goal["category"]]
    T_world_obj = pose7d_to_matrix(goal["pose"])
    any_bbox    = False

    for cam_name, cam in cameras.items():
        lbl_path = OUT_DIR / "labels" / split / f"{env_name}_{cam_name}.txt"

        bbox = (
            compute_yolo_bbox(verts, goal["meshscale"], T_world_obj,
                              cam["K"], cam["T_world_cam"], CAM_WIDTH, CAM_HEIGHT)
            if verts is not None else None
        )

        if bbox is None:
            lbl_path.write_text("")
            stats["no_bbox"] += 1
        else:
            cx, cy, w, h = bbox
            lbl_path.write_text(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
            any_bbox = True

    if any_bbox:
        stats["ok"] += 1

    pct = (env_idx + 1) / n_total * 100
    print(f"  [{pct:5.1f}%] {env_name:<10s}  class={goal['category']:<20s}  split={split}")

print("\n── Done ──────────────────────────────────────────────────────────")
print(f"  Scenes with ≥1 valid bbox : {stats['ok']}")
print(f"  Scenes skipped (no goal)  : {stats['no_goal']}")
print(f"  Scenes skipped (no mesh)  : {stats['no_mesh']}")
print(f"  Camera views with no bbox : {stats['no_bbox']}")
print(f"  Labels written to         : {OUT_DIR / 'labels'}")

  [  1.2%] env_0       class=bed                   split=train
  [  2.5%] env_1       class=lamp                  split=train
  [  3.8%] env_2       class=piano                 split=train
  [  5.0%] env_3       class=vase                  split=train
  [  6.2%] env_4       class=lamp                  split=train
  [  7.5%] env_5       class=bed                   split=train
  [  8.8%] env_6       class=bed                   split=train
  [ 10.0%] env_7       class=airplane              split=train
  [ 11.2%] env_8       class=stairs                split=train
  [ 12.5%] env_9       class=firefighterFigure     split=train
  [ 13.8%] env_10      class=xbox                  split=train
  [ 15.0%] env_11      class=stairs                split=train
  [ 16.2%] env_12      class=cone                  split=train
  [ 17.5%] env_13      class=xbox                  split=train
  [ 18.8%] env_14      class=firefighterFigure     split=train
  [ 20.0%] env_15      class=airplane              spli